# A large unit can hide the neighbor

Distance adds squared gaps. If one measurement is recorded in a huge unit, its gap dominates the sum, even when that gap is unimportant.

Two stored visits, and a query:

| | Feature 1 | Feature 2 | Label |
|---|---:|---:|---|
| A | 0 | 100 | no |
| B | 5 | 101 | yes |
| Q | 0 | 102 | ? |

Squared distances, with no scaling:

$$
\begin{aligned}
d^2(Q, A) &= (0-0)^2 + (102-100)^2 = 4 \\
d^2(Q, B) &= (0-5)^2 + (102-101)^2 = 25 + 1 = 26
\end{aligned}
$$

A is nearer. The answer would be no.

Now give the two features a common size, using only the stored rows. The training means are

$$
\bar x = \frac{0+5}{2} = \frac{5}{2}, \qquad
\bar y = \frac{100+101}{2} = \frac{201}{2}
$$

The root-mean-square spread, dividing by $n$ rather than by $n-1$, is

$$
s_x = \sqrt{\frac{1}{2}\left(\left(-\frac{5}{2}\right)^2 + \left(\frac{5}{2}\right)^2\right)}
= \sqrt{\frac{25}{4}} = \frac{5}{2}
$$

$$
s_y = \sqrt{\frac{1}{2}\left(\left(-\frac{1}{2}\right)^2 + \left(\frac{1}{2}\right)^2\right)}
= \sqrt{\frac{1}{4}} = \frac{1}{2}
$$

The scaled coordinates are $(x - \bar x)/s_x$ and $(y - \bar y)/s_y$.

$$
\begin{aligned}
A &\mapsto (-1,\ -1) \\
B &\mapsto (1,\ 1) \\
Q &\mapsto (-1,\ 3)
\end{aligned}
$$

The query uses the training center and the training spread. It does not get a vote in those two numbers.

$$
\begin{aligned}
d^2(Q, A) &= (-1-(-1))^2 + (3-(-1))^2 = 16 \\
d^2(Q, B) &= (-1-1)^2 + (3-1)^2 = 8
\end{aligned}
$$

B is nearer. The label flips from no to yes. The visits did not change. The units did.


In [1]:
# Raw distances pick A. Training-only scaling picks B.
from fractions import Fraction

raw_a = (0 - 0) ** 2 + (102 - 100) ** 2
raw_b = (0 - 5) ** 2 + (102 - 101) ** 2
assert raw_a == 4 and raw_b == 26 and raw_a < raw_b

mean_x, mean_y = Fraction(5, 2), Fraction(201, 2)
scale_x, scale_y = Fraction(5, 2), Fraction(1, 2)

def place(x, y):
    return ((x - mean_x) / scale_x, (y - mean_y) / scale_y)

scaled = {"A": place(0, 100), "B": place(5, 101), "Q": place(0, 102)}
assert scaled["A"] == (-1, -1)
assert scaled["B"] == (1, 1)
assert scaled["Q"] == (-1, 3)

def squared(p, q):
    return (p[0] - q[0]) ** 2 + (p[1] - q[1]) ** 2

assert squared(scaled["Q"], scaled["A"]) == 16
assert squared(scaled["Q"], scaled["B"]) == 8
print("raw nearer A", raw_a, raw_b)
print("scaled nearer B", squared(scaled["Q"], scaled["A"]), squared(scaled["Q"], scaled["B"]))


raw nearer A 4 26
scaled nearer B 16 8


## A pitfall

Scaling after you have already declared a nearest neighbor is too late. The neighbor was chosen in the old units. Scale the stored rows first, with a rule you can recompute, then measure.

## What to carry forward

Unscaled, $d^2$ is $4$ to A and $26$ to B, so A wins. After training-only scaling, $d^2$ is $16$ to A and $8$ to B, so B wins.
